# Train U-Net from scratch on Kvasir-SEG

This notebook is the practical part of the MN5 training. The goal is to train a small U-Net from scratch for binary polyp segmentation using the local Kvasir-SEG dataset included with the repository.

You will run the same notebook twice inside a Slurm allocation that reserves two GPUs:

1. `requested_gpus = 1` for the single-GPU baseline.
2. `requested_gpus = 2` for the two-GPU comparison.

The notebook is intentionally offline-only. It never downloads data during the session.

## What is U-Net?

U-Net is a convolutional neural network architecture designed for image segmentation. Instead of predicting one label for a whole image, it predicts one label per pixel.

The architecture has three important ideas:

- **Encoder path**: progressively downsamples the image and learns high-level semantic features.
- **Decoder path**: upsamples those features back to the original image resolution.
- **Skip connections**: copy fine spatial details from the encoder into the decoder, helping the model recover precise object boundaries.

For Kvasir-SEG, the input is an RGB endoscopy image and the target is a binary mask where polyp pixels are foreground and the rest is background.

In [ ]:
from pathlib import Path
import os
import random
import time
import warnings
import zipfile

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

SEED = 999

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

DATA_ROOT = Path("data")
DATASET_NAME = "Kvasir-SEG"
ZIP_CANDIDATES = [DATA_ROOT / "kvasir-seg.zip", DATA_ROOT / "Kvasir-SEG.zip"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_size = 256
batch_size = 8
epochs = 5
learning_rate = 1e-3
num_workers = 0  # Keep 0 in Jupyter on MN5 to avoid multiprocessing worker issues.
requested_gpus = 1  # First run: 1. Second run: restart the kernel and set this to 2.

print(f"Seed: {SEED}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Visible GPUs: {torch.cuda.device_count()}")

## Prepare the dataset

The dataset is mandatory and must already be available locally. This notebook accepts either:

- `data/kvasir-seg.zip`
- `data/Kvasir-SEG.zip`
- an extracted directory under `data/` containing `images/` and `masks/`

If something is missing or corrupted, the next cell raises a short, actionable error instead of failing later with a cryptic traceback.

In [ ]:
class TrainingSetupError(RuntimeError):
    pass


def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    try:
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                target = (destination / member).resolve()
                if not str(target).startswith(str(destination)):
                    raise TrainingSetupError(f"Unsafe path found inside zip: {member}")
            archive.extractall(destination)
    except zipfile.BadZipFile as exc:
        raise TrainingSetupError(f"The dataset archive is not a valid zip file: {zip_path}") from exc
    except PermissionError as exc:
        raise TrainingSetupError(f"No permission to extract the dataset into: {destination}") from exc


def find_kvasir_root(data_root):
    if not data_root.exists():
        return None
    candidates = [data_root / DATASET_NAME, data_root / DATASET_NAME.lower()]
    candidates.extend(path for path in data_root.iterdir() if path.is_dir())
    for candidate in candidates:
        if (candidate / "images").is_dir() and (candidate / "masks").is_dir():
            return candidate
    return None


def prepare_dataset():
    DATA_ROOT.mkdir(exist_ok=True)
    dataset_dir = find_kvasir_root(DATA_ROOT)
    if dataset_dir is not None:
        return dataset_dir

    zip_path = next((path for path in ZIP_CANDIDATES if path.exists()), None)
    if zip_path is None:
        expected = " or ".join(str(path) for path in ZIP_CANDIDATES)
        raise TrainingSetupError(
            "Dataset not found. Place the mandatory Kvasir-SEG archive at "
            f"{expected}, or extract it under data/ with images/ and masks/ folders."
        )

    print(f"Using local dataset archive: {zip_path}")
    print("Extracting dataset. This runs only if the extracted folder is missing.")
    safe_extract_zip(zip_path, DATA_ROOT)

    dataset_dir = find_kvasir_root(DATA_ROOT)
    if dataset_dir is None:
        raise TrainingSetupError("Extraction finished, but no images/ and masks/ folders were found under data/.")
    return dataset_dir


try:
    DATASET_DIR = prepare_dataset()
    images_dir = DATASET_DIR / "images"
    masks_dir = DATASET_DIR / "masks"
    image_files = sorted(path for path in images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    mask_files = sorted(path for path in masks_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    if not image_files or not mask_files:
        raise TrainingSetupError("The dataset folders exist, but images or masks are empty.")
    print(f"Dataset root: {DATASET_DIR}")
    print(f"Images: {len(image_files)}")
    print(f"Masks:  {len(mask_files)}")
    print(f"Example image: {image_files[0].name}")
except TrainingSetupError as exc:
    raise RuntimeError(f"Dataset setup failed: {exc}") from exc

In [ ]:
class KvasirSegDataset(Dataset):
    def __init__(self, root, size=256):
        self.root = Path(root)
        self.size = size
        images = {
            path.stem: path
            for path in (self.root / "images").iterdir()
            if path.suffix.lower() in IMAGE_EXTENSIONS
        }
        masks = {
            path.stem: path
            for path in (self.root / "masks").iterdir()
            if path.suffix.lower() in IMAGE_EXTENSIONS
        }
        self.keys = sorted(images.keys() & masks.keys())
        self.images = images
        self.masks = masks
        if not self.keys:
            raise TrainingSetupError("No matching image/mask pairs found. Check file names in images/ and masks/.")

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, index):
        key = self.keys[index]
        try:
            image = Image.open(self.images[key]).convert("RGB").resize((self.size, self.size))
            mask = Image.open(self.masks[key]).convert("L").resize((self.size, self.size), Image.Resampling.NEAREST)
        except Exception as exc:
            raise RuntimeError(f"Could not load image/mask pair: {key}") from exc

        image = np.asarray(image, dtype=np.float32) / 255.0
        mask = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)
        image = torch.from_numpy(image).permute(2, 0, 1)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask


def build_dataloaders(dataset_dir):
    dataset = KvasirSegDataset(dataset_dir, image_size)
    train_len = int(0.8 * len(dataset))
    val_len = len(dataset) - train_len
    if train_len == 0 or val_len == 0:
        raise TrainingSetupError("Dataset is too small for an 80/20 train-validation split.")
    train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))
    loader_kwargs = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    return dataset, train_loader, val_loader


try:
    dataset, train_loader, val_loader = build_dataloaders(DATASET_DIR)
    print(f"Matched pairs: {len(dataset)}")
    print(f"Train batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
except TrainingSetupError as exc:
    raise RuntimeError(f"DataLoader setup failed: {exc}") from exc

In [ ]:
images, masks = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(images[i].permute(1, 2, 0))
    axes[0, i].set_title("image")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks[i, 0], cmap="gray")
    axes[1, i].set_title("mask")
    axes[1, i].axis("off")
plt.tight_layout()

## Define U-Net from scratch

The model below is intentionally small enough for a training session, but it keeps the core U-Net structure:

- two convolutions per block
- max-pooling in the encoder
- transposed convolutions in the decoder
- skip connections between matching encoder and decoder levels

There are no pretrained weights. The model starts from random initialization controlled by `SEED = 999`.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(channels, feature))
            channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        channels = features[-1] * 2
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(channels, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))
            channels = feature

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for index in range(0, len(self.ups), 2):
            x = self.ups[index](x)
            skip = skip_connections[index // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat((skip, x), dim=1)
            x = self.ups[index + 1](x)

        return self.final(x)

## Metrics

Segmentation should not be evaluated only with loss. This notebook reports:

- **Dice score**: overlap metric commonly used in medical segmentation.
- **IoU / Jaccard index**: intersection divided by union.

Both metrics range from 0 to 1, where higher is better.

In [ ]:
def dice_score_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    predictions = (torch.sigmoid(logits) > threshold).float()
    intersection = (predictions * targets).sum(dim=(1, 2, 3))
    total = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (total + eps)).mean()


def iou_score_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    predictions = (torch.sigmoid(logits) > threshold).float()
    intersection = (predictions * targets).sum(dim=(1, 2, 3))
    union = predictions.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    return ((intersection + eps) / (union + eps)).mean()

In [ ]:
def build_model():
    available_gpus = torch.cuda.device_count()
    if requested_gpus < 1:
        raise ValueError("requested_gpus must be 1 or 2 for this training exercise.")

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    use_gpus = min(requested_gpus, available_gpus)
    model = UNet().to(device)

    if use_gpus > 1:
        model = nn.DataParallel(model, device_ids=list(range(use_gpus)))
        print(f"Training with DataParallel on {use_gpus} GPUs")
    elif use_gpus == 1:
        print("Training on one GPU")
    else:
        print("CUDA is not available. Training on CPU for debugging only.")

    return model, device


model, device = build_model()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

In [ ]:
def run_epoch(model, loader, training):
    model.train(training)
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        try:
            with torch.set_grad_enabled(training):
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model(images)
                    loss = criterion(logits, masks)

                if training:
                    optimizer.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                raise RuntimeError(
                    "CUDA ran out of memory. Reduce batch_size or image_size, restart the kernel, and rerun the notebook."
                ) from exc
            raise

        batch_size_now = images.size(0)
        total_loss += loss.item() * batch_size_now
        total_dice += dice_score_from_logits(logits.detach(), masks).item() * batch_size_now
        total_iou += iou_score_from_logits(logits.detach(), masks).item() * batch_size_now

    n = len(loader.dataset)
    return total_loss / n, total_dice / n, total_iou / n


history = []
start = time.perf_counter()

for epoch in range(1, epochs + 1):
    train_loss, train_dice, train_iou = run_epoch(model, train_loader, training=True)
    val_loss, val_dice, val_iou = run_epoch(model, val_loader, training=False)
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_dice": train_dice,
        "train_iou": train_iou,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
    }
    history.append(row)
    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} dice {train_dice:.4f} iou {train_iou:.4f} | "
        f"val loss {val_loss:.4f} dice {val_dice:.4f} iou {val_iou:.4f}"
    )

elapsed = time.perf_counter() - start
print(f"Total training time: {elapsed:.1f} seconds")

In [ ]:
epochs_axis = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs_axis, [row["train_loss"] for row in history], label="train")
axes[0].plot(epochs_axis, [row["val_loss"] for row in history], label="validation")
axes[0].set_title("Loss")
axes[1].plot(epochs_axis, [row["train_dice"] for row in history], label="train")
axes[1].plot(epochs_axis, [row["val_dice"] for row in history], label="validation")
axes[1].set_title("Dice")
axes[2].plot(epochs_axis, [row["train_iou"] for row in history], label="train")
axes[2].plot(epochs_axis, [row["val_iou"] for row in history], label="validation")
axes[2].set_title("IoU")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()

In [ ]:
model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad():
    logits = model(images.to(device))
predictions = (torch.sigmoid(logits).cpu() > 0.5).float()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0, i].imshow(images[i].permute(1, 2, 0))
    axes[0, i].set_title("image")
    axes[1, i].imshow(masks[i, 0], cmap="gray")
    axes[1, i].set_title("mask")
    axes[2, i].imshow(predictions[i, 0], cmap="gray")
    axes[2, i].set_title("prediction")
    for ax in axes[:, i]:
        ax.axis("off")
plt.tight_layout()

## Exercise

1. Run the notebook with `requested_gpus = 1` and record total training time, validation Dice, and validation IoU.
2. Restart the kernel, set `requested_gpus = 2`, and rerun the notebook.
3. Compare speed and validation quality between the two runs.

## Challenge

Implement one additional segmentation metric beyond Dice and IoU.

Good options are:

- pixel accuracy
- precision and recall
- F1 score computed from true positives, false positives, and false negatives
- boundary-based error

Add your metric to the training loop, store it in `history`, and plot it next to Dice and IoU.